In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import annoy

from plotnine import *
import matplotlib.pyplot as plt
import seaborn as sb
from collections import Counter


sc.set_figure_params(figsize=(5,5)) # no blurry figures allowed

pd.set_option('display.max_columns', None)

In [ ]:
dr = "/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC/"

# Load data

In [ ]:
mice1 = sc.read_h5ad(dr + '19.11.25.mice1_exe_qc_filtercells.h5ad')
mice1
mice2 = sc.read_h5ad(dr + '2.12.25.mice2_exe_qc_filtercells.h5ad')
mice2
mice3 = sc.read_h5ad(dr + '19.1.26.mice3_exe_qc_filtercells.h5ad')
mice3
mice4 = sc.read_h5ad(dr + '19.1.26.mice4_exetr_qc_filtercells.h5ad')
mice4
mice5 = sc.read_h5ad(dr + '19.1.26.mice5_exetr_qc_filtercells.h5ad')
mice5
mice6 = sc.read_h5ad(dr + '19.1.26.mice6_exetr_qc_filtercells.h5ad')
mice6
mice7 = sc.read_h5ad(dr + '19.1.26.mice7_exetr_qc_filtercells.h5ad')
mice7

In [ ]:
adatas = [mice1,mice2,mice3,mice4,mice5,mice6,mice7]
batches = ['Mice1','Mice2','Mice3','Mice4','Mice5','Mice6','Mice7'
          ]
adata = adatas[0].concatenate(adatas[1:], join='inner', batch_categories=batches, batch_key='mouse_id')
adata

In [ ]:
from collections import Counter
Counter(adata.obs['mouse_id'])

In [ ]:
# map each mouse_id to a condition
cond_map = {
    'Mice1': 'sedentary',
    'Mice2': 'sedentary',
    'Mice3': 'sedentary',
    'Mice4': 'exercise',
    'Mice5': 'exercise',
    'Mice6': 'exercise',
    'Mice7': 'exercise',
}

adata.obs['condition'] = adata.obs['mouse_id'].map(cond_map).astype('category')

In [ ]:
adata.obs["mouse_id"] = (
    adata.obs["mouse_id"]
    .str.replace("Mice", "Mouse", regex=False)
)

In [ ]:
pd.crosstab(adata.obs["condition"], adata.obs["mouse_id"],margins = True)

In [ ]:
# try out gene thresholds
sum(sc.pp.filter_genes(adata, min_cells=30, inplace=False)[0])

In [ ]:
print(adata.shape)

In [ ]:
sc.pp.filter_genes(adata, min_cells=30, inplace=True)

In [ ]:
print(adata.shape)

In [ ]:
adata.layers["raw_counts"]  = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, exclude_highly_expressed = True)
adata.layers["norm_counts"]  = adata.X.copy()
sc.pp.log1p(adata)
adata.layers["norm_counts_log"]  = adata.X.copy()
adata

In [ ]:
sc.tl.pca(adata, n_comps=100, svd_solver='arpack')
adata_no_harmony = adata.copy()

In [ ]:
sc.external.pp.harmony_integrate(adata, key='mouse_id')
n_pcs = 50
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=n_pcs, use_rep='X_pca_harmony', metric="cosine") 
sc.tl.leiden(adata, resolution=0.8) # A higher resolution parameter leads to more clusters.
sc.tl.umap(adata)

In [ ]:
# with integration
sc.pl.umap(adata, color=['leiden','condition','mouse_id',"Il15", "Hba-a1", "Hba-a2", "Hbb-bs", "Hbb-bt"
                         ],
           ncols= 2,  vmin='p5', vmax='p95', wspace = 0.3)

# Removing Contaminants

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=["Hba-a1", "Hba-a2", "Hbb-bs", "Hbb-bt"],
    groupby="leiden"
)

In [ ]:
hb_genes = ["Hba-a1", "Hba-a2", "Hbb-bs", "Hbb-bt"]
hb_present = [g for g in hb_genes if g in adata.var_names]
adata = adata[:, ~adata.var_names.isin(hb_present)].copy()

print("Removed:", hb_present)

In [ ]:
#re run the normalization, pca, harmony

In [ ]:
# with integration
sc.pl.umap(adata, color=['leiden','condition','mouse_id',"Il15",
                         ],
           ncols= 2,  vmin='p5', vmax='p95', wspace = 0.3)

In [ ]:
adata.layers['norm_counts_log_dense'] = adata.layers['norm_counts_log'].todense()
adata.layers['raw_counts_dense'] = adata.layers['raw_counts'].todense()

# Celltype markers

In [ ]:
# ==============================
# Mouse Skeletal Muscle scRNA-seq
# More Representative Cell Type Marker Panel
# ==============================

marker_dict = {

    # --------------------------
    # Skeletal muscle / myonuclei
    # --------------------------
    "Muscle_Myonuclei_Mature": [
        "Acta1", "Ckm", "Ttn",
        "Myh1", "Myh2", "Myh4",
        "Tnnt3", "Tnni2", "Tnnc2",
        "Myl1", "Mylpf", "Mylk2",
        "Des", "Tpm1"
    ],
    # Slow / oxidative myonuclei markers (often lower in some preps)
    "Muscle_Slow_Oxidative": ["Myh7", "Tnnt1", "Tnnc1", "Myl2"],

    # Fiber type helpers (single genes are noisy; keep as cues)
    "Fiber_Type_2B": ["Myh4"],
    "Fiber_Type_2X": ["Myh1"],
    "Fiber_Type_2A": ["Myh2"],
    "Fiber_Type_1_Slow": ["Myh7", "Tnnt1", "Myl2"],

    # --------------------------
    # Myogenic progenitors
    # --------------------------
    "Satellite_Quiescent": [
        "Pax7", "Vcam1", "Itga7", "Sdc4",
        "Cdh15", "Spry1", "Notch1"
    ],
    "Satellite_Activated_Myoblast": [
        "Myod1", "Myf5", "Des",
        "Mki67", "Top2a", "Pcna"
    ],

    # --------------------------
    # FAP / stromal
    # --------------------------
    "FAP_Core": ["Pdgfra", "Dcn", "Col1a1", "Col1a2", "Lum", "Cd34", "Ly6a"],
    "FAP_ECM_Remodeling": ["Col3a1", "Cthrc1", "Postn", "Fn1", "Timp1"],
    # FAP sub-states you’ve been seeing
    "FAP_Prg4+": ["Prg4"],
    "FAP_Cxcl14+": ["Cxcl14"],
    "FAP_Dpt+": ["Dpt"],
    "FAP_Apoe_Il33": ["Apoe", "Il33"],

    # Tendon / tenocyte-like (can appear as contamination or real MTJ region)
    "Tenocyte": ["Scx", "Tnmd", "Col1a1", "Col3a1", "Tnc"],

    # --------------------------
    # Vascular
    # --------------------------
    "Endothelial": ["Pecam1", "Cdh5", "Kdr", "Tek", "Vwf", "Emcn", "Esam"],
    # Lymphatic endothelium (optional, only if present)
    "Endothelial_Lymphatic": ["Lyve1", "Pdpn", "Prox1"],

    # Pericytes vs Smooth muscle (split!)
    "Pericyte": ["Rgs5", "Pdgfrb", "Cspg4", "Des", "Myl9"],
    "Smooth_Muscle_Vascular": ["Myh11", "Tagln", "Cnn1", "Acta2", "Smtn"],

    # --------------------------
    # Neural / glial (in muscle = Schwann)
    # --------------------------
    "Schwann_Glial": ["Sox10", "Plp1", "Mpz", "Pmp22", "Mbp", "Ngfr"],

    # --------------------------
    # Immune overview
    # --------------------------
    "Immune_All": ["Ptprc"],

    "Macrophage_Monocyte": ["Lyz2", "Csf1r", "Adgre1", "Itgam", "Cd68"],
    "Dendritic": ["Itgax", "Flt3", "Clec9a"],

    "T_cell": ["Cd3d", "Cd3e", "Trbc1", "Trbc2", "Il7r"],
    "T_cell_Cytotoxic": ["Nkg7", "Gzmb", "Prf1"],

    "NK_cell": ["Nkg7", "Ncr1", "Eomes", "Prf1"],
    "B_cell": ["Cd79a", "Ms4a1", "Cd74", "H2-Aa"],
    "Plasma_cell": ["Jchain", "Mzb1", "Xbp1"],

    "Neutrophil": ["S100a8", "S100a9", "Ly6g", "Mpo"],
    "Mast_cell": ["Kit", "Tpsb2", "Gata2"],

    # --------------------------
    # Blood contaminants
    # --------------------------
    "Erythrocyte_RBC": ["Alas2"],
    "Platelet": [ "Pf4", "Itga2b"],

    # --------------------------
    # Adipocytes (if any)
    # --------------------------
    "Adipocyte": ["Adipoq", "Plin1", "Fabp4", "Lpl"],
}

# Example plotting
sc.pl.dotplot(adata, marker_dict, groupby="leiden", standard_scale="var", swap_axes=True)
sc.pl.matrixplot(adata, marker_dict, groupby="leiden", standard_scale="var", swap_axes=True)

In [ ]:
# ============================================
# Leiden cluster -> cell-type marker testing (UPDATED)
# Mouse skeletal muscle scRNA-seq
# ============================================

# --------- SETTINGS ----------
LEIDEN_KEY = "leiden"
LAYER = "norm_counts_log"      # set to None if using adata.X
USE_RAW = False

# --------- UPDATED MARKERS (clean + specific) ----------
MARKERS = {

    # Muscle stem cells
    "MuSC": ["Pax7", "Myod1", "Vcam1", "Sdc4"],

    # Fibro-adipogenic progenitors (core stromal)
    "FAP_core": ["Pdgfra", "Dcn", "Ly6a", "Cd34"],

    # FAP states (Yang et al.)
    "FAP_Cxcl14+": ["Cxcl14"],
    "FAP_Prg4+": ["Prg4"],
    "MAB_Alpl+": ["Alpl"],
    "FAP_Sca1+_Dpt": ["Dpt"],
    "FAP_Apoe_Il33": ["Apoe", "Il33"],

    # Endothelial (strong core vascular markers)
    "Endothelial": ["Pecam1", "Cdh5", "Kdr", "Tek", "Vwf"],

    # Smooth Muscle (true vascular SMC — NOT activated FAPs)
    "Smooth_Muscle": ["Myh11", "Tagln", "Cnn1", "Acta2"],

    # Pericyte (often confused with SMC)
    "Pericyte": ["Rgs5", "Pdgfrb", "Myl9"],

    # Macrophage / Monocyte
    "Macrophage": ["Adgre1", "Csf1r", "Lyz2"],

    # Neutrophil
    "Neutrophil": ["S100a8", "S100a9", "Ly6g"],

    # T cell
    "T_cell": ["Cd3d", "Cd3e", "Cd8a", "Cd4"],

    # B cell
    "B_cell": ["Cd19", "Ms4a1"],

    # Schwann / Glial (peripheral nerve in muscle)
    "Glial_Schwann": ["Sox10", "Plp1", "Mpz", "Pmp22", "Mbp"],
}

# Flatten gene list
all_markers = sorted({g for genes in MARKERS.values() for g in genes})

# --------- Restrict to genes present ----------
def present_genes(adata, genes):
    varnames = adata.raw.var_names if (USE_RAW and adata.raw is not None) else adata.var_names
    genes_present = [g for g in genes if g in varnames]
    genes_missing = [g for g in genes if g not in varnames]
    return genes_present, genes_missing

genes_present, genes_missing = present_genes(adata, all_markers)

print(f"Total requested markers: {len(all_markers)}")
print(f"Markers present in adata: {len(genes_present)}")

if genes_missing:
    print("\nMissing markers:")
    print(", ".join(genes_missing))

markers_present_dict = {}
for ct, genes in MARKERS.items():
    gp, _ = present_genes(adata, genes)
    if len(gp) > 0:
        markers_present_dict[ct] = gp

# --------- 1) Dotplot ----------
sc.pl.dotplot(
    adata,
    var_names=markers_present_dict,
    groupby=LEIDEN_KEY,
    layer=LAYER,
    use_raw=USE_RAW,
    standard_scale="var",
    swap_axes=True,
)

# --------- 2) Heatmap ----------
sc.pl.heatmap(
    adata,
    var_names=markers_present_dict,
    groupby=LEIDEN_KEY,
    layer=LAYER,
    use_raw=USE_RAW,
    standard_scale="var",
    swap_axes=True,
)

# --------- 3) Gene set scoring ----------
for ct, genes in markers_present_dict.items():
    sc.tl.score_genes(
        adata,
        gene_list=genes,
        score_name=f"score_{ct}",
        use_raw=USE_RAW,
        layer=LAYER
    )

score_cols = [c for c in adata.obs.columns if c.startswith("score_")]

scores_by_leiden = (
    adata.obs[[LEIDEN_KEY] + score_cols]
    .groupby(LEIDEN_KEY)
    .mean()
    .sort_index()
)

display(scores_by_leiden)

# Cluster-level heatmap of scores
sc.pl.heatmap(
    adata,
    var_names=score_cols,
    groupby=LEIDEN_KEY,
    swap_axes=True,
)

# --------- 4) Top label guess ----------
top_label = scores_by_leiden.idxmax(axis=1).str.replace("score_", "", regex=False)
top_label_df = pd.DataFrame({
    "leiden": scores_by_leiden.index,
    "top_marker_set": top_label.values
})

display(top_label_df)

In [ ]:
cluster_to_celltype = {
    "0": "FAP",
    "1": "FAP",
    "2": "FAP",
    "3": "FAP",
    "4": "FAP",
    "5": "FAP",
    "6": "FAP",
    "7": "T_cell",
    "8": "FAP",
    "9": "FAP",
    "10": "Tenocyte",
    "11": "FAP",
    "12": "FAP",
    "13": "Glial",
    "14": "Neutrophils",
    "15": "B_cell",
    "16": "Satellite_cell",
    "17": "Macrophage",
    "18": "FAP",
    "19": "Muscle_fiber"
}

adata.obs["leiden"] = adata.obs["leiden"].astype(str)
adata.obs["celltype"] = adata.obs["leiden"].map(cluster_to_celltype)
adata.obs["celltype"] = adata.obs["celltype"].astype("category")

In [ ]:
sc.pl.umap(adata, color=['leiden','condition','mouse_id',"celltype", "Il15", "Il15ra"
                         ],legend_loc='on data', 
           ncols= 2,  vmin='p5', vmax='p95', wspace = 0.3)

In [ ]:
sc.pl.umap(adata, color=["celltype","condition", "Il15", "Il15ra",
                         ],legend_loc='on data',       ncols= 2,  vmin='p5', vmax='p95', wspace = 0.3)

# FAP celltype clustering

In [ ]:
fap_clusters = ["0","1","2","3","4","5","6","8","18","11","12"]

adata_fap = adata[adata.obs["leiden"].isin(fap_clusters)].copy()

In [ ]:
adata_fap.X = adata_fap.layers["raw_counts"].copy()

In [ ]:
sc.pp.normalize_total(adata_fap, exclude_highly_expressed = True)
adata_fap.layers["norm_counts"]  = adata_fap.X.copy()
sc.pp.log1p(adata_fap)
adata_fap.layers["norm_counts_log"]  = adata_fap.X.copy()
adata_fap

In [ ]:
sc.tl.pca(adata_fap, n_comps=100, svd_solver='arpack')

In [ ]:
sc.external.pp.harmony_integrate(adata_fap, key='mouse_id')
n_pcs = 50
sc.pp.neighbors(adata_fap, n_neighbors=30, n_pcs=n_pcs, use_rep='X_pca_harmony', metric="cosine") 
sc.tl.leiden(adata_fap, resolution=0.8) # A higher resolution parameter leads to more clusters.
sc.tl.umap(adata_fap)

In [ ]:
#genes i found from Kellis paper excel of top lfc genes in each Fapsubtypemarker_dict 
Marker_dict= {
    "IPC_SkM": ["Dpp4", "Pi16"],
    "FAP_Cxcl14": ["Cxcl14"],
    "FAP_Prg4": ["Prg4"],
    "FAP_CD142": ["F3"],
    "FAP_post_injury": ["Il33", "Apoe"],
    "MAB": ["Alpl"],
    "Sca1_high": ["Ly6a"]
}
sc.pl.umap(
    adata_fap,
    color=[
        "leiden",
        "Prg4", "F3", "Alpl", "Ly6a", "Pdgfra", "Cd34", "Cxcl14","Il33", "Apoe", "Apod", "Pi16","Postn", "Cfh"
    ],
    ncols=2,
    vmin="p5",
    vmax="p95",
    wspace=0.3
)

In [ ]:
import scanpy as sc
marker_dict= {
    "IPC_SkM": ["Dpp4", "Pi16"],
    "FAP_Cxcl14": ["Cxcl14"],
    "FAP_Prg4": ["Prg4"],
    "FAP_CD142": ["F3"],
    "FAP_post_injury": ["Il33", "Apoe"],
    "MAB": ["Alpl"],
    "Sca1_high": ["Ly6a"]
}
sc.pl
for subtype, genes in marker_dict.items():
    sc.tl.score_genes(
        adata_fap,
        gene_list=genes,
        score_name=subtype,
        use_raw=False,
    
    )

sc.pl.umap(adata_fap, color=list(marker_dict.keys()))

In [ ]:
marker_check = ["Dpp4", "Pi16","Cxcl14","Prg4","F3","Il33", "Apoe","Alpl","Ly6a"]

for gene in marker_check:
    sc.pl.dotplot(adata_fap, gene, groupby = "leiden")

# Comparing each Fap cluster v the other to confirm markers

In [ ]:
# Ensure leiden is string
adata_fap.obs["leiden"] = adata_fap.obs["leiden"].astype(str)

# Create binary columns for clusters 1 through 10
for i in range(0, 11):
    col_name = f"cluster{i}_vs_other"
    adata_fap.obs[col_name] = (adata_fap.obs["leiden"] == str(i)).astype(int)
    
    # Print counts for each
    print(f"{col_name} value counts:")
    print(adata_fap.obs[col_name].value_counts())
    print("-" * 30)

In [ ]:
lfcs_cluster0_or_not = new_lfcs(
    adata_fap,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster0_vs_other"
)

lfcs_cluster1_or_not = new_lfcs(
    adata_fap,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster1_vs_other"
)

lfcs_cluster2_or_not = new_lfcs(
    adata_fap,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster2_vs_other"
)

lfcs_cluster3_or_not = new_lfcs(
    adata_fap,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster3_vs_other"
)

lfcs_cluster4_or_not = new_lfcs(
    adata_fap,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster4_vs_other"
)

lfcs_cluster5_or_not = new_lfcs(
    adata_fap,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster5_vs_other"
)

lfcs_cluster6_or_not = new_lfcs(
    adata_fap,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster6_vs_other"
)

lfcs_cluster7_or_not = new_lfcs(
    adata_fap,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster7_vs_other"
)

lfcs_cluster8_or_not = new_lfcs(
    adata_fap,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster8_vs_other"
)

lfcs_cluster9_or_not = new_lfcs(
    adata_fap,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster9_vs_other"
)

lfcs_cluster10_or_not = new_lfcs(
    adata_fap,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster10_vs_other"
)

In [ ]:
fapcluster_to_celltype = {
    "0": "Prg4+",
    "1": "IPC_Skm",
    "2": "Cxcl14+",
    "3": "Cxcl14+",
    "4": "Cxcl14+",
    "5": "CD142+",
    "6": "FAP",
    "7": "IPC_Skm",
    "8": "Prg4+",
    "9": "Cxcl14+",
    "10": "Sca1-",
   
}

adata_fap.obs["leiden"] = adata_fap.obs["leiden"].astype(str)
adata_fap.obs["celltype"] = adata_fap.obs["leiden"].map(fapcluster_to_celltype)
adata_fap.obs["celltype"] = adata_fap.obs["celltype"].astype("category")

In [ ]:
sc.pl.umap(adata_fap, color=['leiden','condition','mouse_id',"celltype", "Il15", "Il15ra", "Il33"
                         ],legend_loc='on data', 
           ncols= 2,  vmin='p5', vmax='p95', wspace = 0.3)

In [ ]:
sc.pl.umap(adata_fap, color=["celltype", "Il15", "Il15ra","Il2rb","Il2rg", "Il33"
                         ],legend_loc='on data', 
           ncols= 2,  vmin='p5', vmax='p95', wspace = 0.3)

In [ ]:
genes_to_plot = ["Il15", "Il15ra", "Il33", "Il2rb", "Il2rg"]

sc.pl.violin(
    adata_fap,
    keys=genes_to_plot,
    groupby="celltype",
    rotation=45,
    stripplot=False,
    jitter=False,
    multi_panel=True
)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import sparse

GENES = ["Il15", "Il15ra", "Il33"]
GROUP_COL = "celltype"
COND_COL = "condition"
LAYER = "norm_counts_log"

# ---------------------------
# Extract expression
# ---------------------------
expr = adata_fap[:, GENES].layers[LAYER]

if sparse.issparse(expr):
    expr = expr.toarray()
else:
    expr = np.asarray(expr)

expr_df = pd.DataFrame(expr, columns=GENES, index=adata_fap.obs_names)

plot_df = pd.concat(
    [adata_fap.obs[[GROUP_COL, COND_COL]], expr_df],
    axis=1
)

plot_df = plot_df.dropna(subset=[GROUP_COL])

# ---------------------------
# 1️⃣ Overall violin (by celltype)
# ---------------------------
for gene in GENES:
    plt.figure(figsize=(10, 5))
    sns.violinplot(
        data=plot_df,
        x=GROUP_COL,
        y=gene,
        cut=0,
        inner="box"
    )
    plt.xticks(rotation=45, ha="right")
    plt.title(f"{gene} expression by celltype")
    plt.tight_layout()
    plt.show()

# Il33 between Celltypes and conditions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Build dataframe
df = pd.DataFrame({
    "Il33": adata_fap[:, ["Il33"]].X.toarray().flatten(),
    "celltype": adata_fap.obs["celltype"].values,
    "condition": adata_fap.obs["condition"].values
})

# Mean expression table
mean_table = df.groupby(["celltype", "condition"])["Il33"].mean().unstack()

# Keep a consistent order
celltypes = mean_table.index.tolist()

# Statistical testing
pvals = []

for ct in celltypes:
    sub = df[df["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"]["Il33"]
    sed = sub[sub["condition"] == "sedentary"]["Il33"]

    if len(ex) > 0 and len(sed) > 0:
        _, p = mannwhitneyu(ex, sed, alternative="two-sided")
    else:
        p = np.nan

    pvals.append(p)

# Adjust p-values
valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = mean_table.join(stats_df)
print(results)

# Plot
ax = mean_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel("Mean Il33 expression")
plt.xlabel("FAP subtype")
plt.title("Mean Il33 expression by cell type and condition")
plt.xticks(rotation=45, ha="right")

# Add significance labels
ymax = mean_table.max(axis=1)
offset = ymax.max() * 0.01  # dynamic spacing

for i, ct in enumerate(mean_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import fisher_exact
threshold = 0.5

expr = adata_fap[:, "Il33"].layers["norm_counts_log"].toarray().flatten()

df = pd.DataFrame({
    "Il33": expr,
    "celltype": adata_fap.obs["celltype"].values,
    "condition": adata_fap.obs["condition"].values
})

df["Il33_pos"] = df["Il33"] > threshold

prop_table = df.groupby(["celltype", "condition"])["Il33_pos"].mean().unstack()
celltypes = prop_table.index.tolist()

pvals = []

for ct in celltypes:
    sub = df[df["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"]["Il33_pos"]
    sed = sub[sub["condition"] == "sedentary"]["Il33_pos"]

    table = np.array([
        [ex.sum(), (~ex).sum()],
        [sed.sum(), (~sed).sum()]
    ])

    if table.sum() > 0:
        _, p = fisher_exact(table)
    else:
        p = np.nan

    pvals.append(p)

valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = prop_table.join(stats_df)

ax = prop_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel("Proportion of Il33+ cells (>0.5)")
plt.xlabel("FAP subtype")
plt.title("Proportion of Il33+ cells by FAP subtype and condition")
plt.xticks(rotation=45, ha="right")

ymax = prop_table.max(axis=1)
offset = max(0.005, ymax.max() * 0.01)

for i, ct in enumerate(prop_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Build dataframe
df = pd.DataFrame({
    "Il15ra": adata_fap[:, ["Il15ra"]].X.toarray().flatten(),
    "celltype": adata_fap.obs["celltype"].values,
    "condition": adata_fap.obs["condition"].values
})

# Mean expression table
mean_table = df.groupby(["celltype", "condition"])["Il15ra"].mean().unstack()

# Keep a consistent order
celltypes = mean_table.index.tolist()

# Statistical testing
pvals = []

for ct in celltypes:
    sub = df[df["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"]["Il15ra"]
    sed = sub[sub["condition"] == "sedentary"]["Il15ra"]

    if len(ex) > 0 and len(sed) > 0:
        _, p = mannwhitneyu(ex, sed, alternative="two-sided")
    else:
        p = np.nan

    pvals.append(p)

# Adjust p-values
valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = mean_table.join(stats_df)
print(results)

# Plot
ax = mean_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel("Mean Il15ra expression")
plt.xlabel("FAP subtype")
plt.title("Mean Il15ra expression by cell type and condition")
plt.xticks(rotation=45, ha="right")

# Add significance labels
ymax = mean_table.max(axis=1)
offset = ymax.max() * 0.01  # dynamic spacing

for i, ct in enumerate(mean_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
#Il15

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Build dataframe
df = pd.DataFrame({
    "Il15": adata_fap[:, ["Il15"]].X.toarray().flatten(),
    "celltype": adata_fap.obs["celltype"].values,
    "condition": adata_fap.obs["condition"].values
})

# Mean expression table
mean_table = df.groupby(["celltype", "condition"])["Il15"].mean().unstack()

# Keep a consistent order
celltypes = mean_table.index.tolist()

# Statistical testing
pvals = []

for ct in celltypes:
    sub = df[df["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"]["Il15"]
    sed = sub[sub["condition"] == "sedentary"]["Il15"]

    if len(ex) > 0 and len(sed) > 0:
        _, p = mannwhitneyu(ex, sed, alternative="two-sided")
    else:
        p = np.nan

    pvals.append(p)

# Adjust p-values
valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = mean_table.join(stats_df)
print(results)

# Plot
ax = mean_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel("Mean Il15 expression")
plt.xlabel("FAP subtype")
plt.title("Mean Il15 expression by cell type and condition")
plt.xticks(rotation=45, ha="right")

# Add significance labels
ymax = mean_table.max(axis=1)
offset = ymax.max() * 0.01  # dynamic spacing

for i, ct in enumerate(mean_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.5 # you can test 0.25, 0.5 as well

df = pd.DataFrame({
    "Il15": adata_fap[:, ["Il15"]].X.toarray().flatten(),
    "celltype": adata_fap.obs["celltype"].values,
    "condition": adata_fap.obs["condition"].values
})

df["Il15_pos"] = df["Il15"] > threshold

prop_table = df.groupby(["celltype", "condition"])["Il15_pos"].mean().unstack()
celltypes = prop_table.index.tolist()

pvals = []

for ct in celltypes:
    sub = df[df["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"]["Il15_pos"]
    sed = sub[sub["condition"] == "sedentary"]["Il15_pos"]

    table = np.array([
        [ex.sum(), (~ex).sum()],
        [sed.sum(), (~sed).sum()]
    ])

    if table.shape == (2, 2):
        _, p = fisher_exact(table)
    else:
        p = np.nan

    pvals.append(p)

valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

def p_to_star(p):
    if pd.isna(p):
        return "na"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = prop_table.join(stats_df)
print(results)

ax = prop_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel(f"Proportion of Il15+ cells (>{threshold})")
plt.xlabel("FAP subtype")
plt.title(f"Proportion of Il15+ cells by FAP subtype and condition (threshold={threshold})")
plt.xticks(rotation=45, ha="right")

ymax = prop_table.max(axis=1)
offset = ymax.max() * 0.005

for i, ct in enumerate(prop_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.5

expr = adata_fap[:, "Il15ra"].layers["norm_counts_log"].toarray().flatten()

df = pd.DataFrame({
    "Il15ra": expr,
    "celltype": adata_fap.obs["celltype"].values,
    "condition": adata_fap.obs["condition"].values
})

df["Il15ra_pos"] = df["Il15ra"] > threshold

prop_table = df.groupby(["celltype", "condition"])["Il15ra_pos"].mean().unstack()
celltypes = prop_table.index.tolist()

pvals = []

for ct in celltypes:
    sub = df[df["celltype"] == ct]

    ex = sub[sub["condition"] == "exercise"]["Il15ra_pos"]
    sed = sub[sub["condition"] == "sedentary"]["Il15ra_pos"]

    table = np.array([
        [ex.sum(), (~ex).sum()],
        [sed.sum(), (~sed).sum()]
    ])

    if table.sum() > 0:
        _, p = fisher_exact(table)
    else:
        p = np.nan

    pvals.append(p)

valid_mask = ~np.isnan(pvals)
padj = np.full(len(pvals), np.nan)
padj[valid_mask] = multipletests(np.array(pvals)[valid_mask], method="fdr_bh")[1]

stats_df = pd.DataFrame({
    "celltype": celltypes,
    "pval": pvals,
    "padj": padj
}).set_index("celltype")

stats_df["sig"] = stats_df["padj"].apply(p_to_star)

results = prop_table.join(stats_df)

ax = prop_table[["exercise", "sedentary"]].plot(kind="bar", figsize=(12, 6))
plt.ylabel("Proportion of Il15ra+ cells (>0.5)")
plt.xlabel("FAP subtype")
plt.title("Proportion of Il15ra+ cells by FAP subtype and condition")
plt.xticks(rotation=45, ha="right")

ymax = prop_table.max(axis=1)
offset = max(0.002, ymax.max() * 0.002)

for i, ct in enumerate(prop_table.index):
    ax.text(i, ymax.loc[ct] + offset, results.loc[ct, "sig"],
            ha="center", va="bottom", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Do FAPs express IL15 receptor components?
genes = ["Il15", "Il15ra", "Il2rb", "Il2rg"]
sc.pl.violin(adata_fap, keys=genes, groupby="celltype", rotation=45)

# Immune Cell subtypes

In [ ]:
immune_types = ['7','14','15', '17']

adata_immune = adata[adata.obs["leiden"].isin(immune_types)].copy()

In [ ]:
print(adata_immune.obs["condition"].value_counts())

In [ ]:
sc.pp.normalize_total(adata_immune, exclude_highly_expressed = True)
adata_immune.layers["norm_counts"]  = adata_immune.X.copy()
sc.pp.log1p(adata_immune)
adata_immune.layers["norm_counts_log"]  = adata_immune.X.copy()
adata_immune

In [ ]:
sc.tl.pca(adata_immune, n_comps=100, svd_solver='arpack')
adata_immune_no_harmony = adata_immune.copy()

In [ ]:
sc.external.pp.harmony_integrate(adata_immune, key='mouse_id')
n_pcs = 50
sc.pp.neighbors(adata_immune, n_neighbors=30, n_pcs=n_pcs, use_rep='X_pca_harmony', metric="cosine") 
sc.tl.leiden(adata_immune, resolution=0.8) # A higher resolution parameter leads to more clusters.
sc.tl.umap(adata_immune)

In [ ]:
Immune_Markers = {

    "B_Cells": ["Cd19", "Cd22", "Ms4a1"],

    "T_Cells": ["Cd3d", "Cd3e", "Cd3g", "Cd8a", "Cd4"],
    "NK_cell": ["Nkg7", "Ncr1", "Eomes", "Prf1"],

    "Macrophages": ["Itgam", "Csf1r", "Adgre1", "Itgb1"],
    "Mast_cell": ["Kit", "Tpsb2", "Gata2"],
    "Dendritic": ["Itgax", "Flt3", "Clec9a"],

    "Neutrophils": ["Itgam", "Cd14", "S100a8", "S100a9"]
}

# Dotplot
sc.pl.dotplot(
    adata_immune,
    Immune_Markers,
    groupby="leiden",
    standard_scale="var",
    swap_axes=True
)

# Matrixplot
sc.pl.matrixplot(
    adata_immune,
    Immune_Markers,
    groupby="leiden",
    standard_scale="var",
    swap_axes=True
)



In [ ]:
immunecluster_to_celltype = {
    "0": "NK_cell",
    "1": "Neutrophil",
    "3": "CD4_Tcell",
    "2": "B_Cell",
    "4": "CD8T_cell",
    "5": "T_cell",
    "6": "Macrophage",
    "7": "Unknown1",
    "8": "Unknown2",
    "9": "Macrophage",
    "10": "NK_cell",
    "11": "Mast_cell",
   

}

adata_immune.obs["leiden"] = adata_immune.obs["leiden"].astype(str)
adata_immune.obs["celltype"] = adata_immune.obs["leiden"].map(immunecluster_to_celltype)
adata_immune.obs["celltype"] = adata_immune.obs["celltype"].astype("category")

In [ ]:
sc.pl.umap(adata_immune, color=['leiden','condition','mouse_id',"celltype", "Il15", "Il15ra", "Il2ra"
                         ],legend_loc='on data', 
           ncols= 2,  vmin='p5', vmax='p95', wspace = 0.3)

# Identifying cluster 6 and 7

In [ ]:
from plotnine import *
from scipy.cluster.hierarchy import dendrogram, linkage, set_link_color_palette
from scipy.stats import gmean, mannwhitneyu, spearmanr

import matplotlib as mpl

cmap = mpl.cm.get_cmap("magma_r")(np.linspace(0,1,30))
magma_cmap = mpl.colors.ListedColormap(cmap[2:,:-1])

import statsmodels.stats.multitest  as mt

from scipy.stats import gmean, mannwhitneyu
import statsmodels.stats.multitest  as mt
def new_lfcs(ad, raw_layer, norm_layer, cl_level = "leiden", pseudocount_threshold = 0.05, cl_pct = 0.15,
              padj_threshold = 0.05, log2FC_threshold = 0.5, alt = "two-sided"):

    # Filter the ad object to include only the desired cl_level values
    # ad = ad[ad.obs[cl_level].isin(["3", "9", "10", "12", "13", "14"])]
    # ad = ad[(ad.obs.leiden == "7")]
    # ad = ad.obs['leiden'] == '0'
    # combined_adata[(combined_adata.obs.leiden == "6")

    def calculate_size_factor(samp, gene_is = ad.var_names):
        g_means = gmean([np.array(ad[samp,gene_is].layers[raw_layer].sum(axis = 0).squeeze()),
                         np.array(ad[~samp,gene_is].layers[raw_layer].sum(axis = 0).squeeze())])
        return np.nanmedian(np.array(ad[samp,gene_is].layers[raw_layer].sum(axis = 0).squeeze())/g_means)

    df = pd.DataFrame()

    empty_genes = ad.var_names[np.where(np.array(np.sum(ad.layers[raw_layer] != 0, axis = 0) == 0).flatten())]
    print("num genes with no counts in raw matrix: %s"%empty_genes.shape[0])
    good_genes = ad.var_names[~ad.var_names.isin(empty_genes)]
    print(good_genes.shape)
    print(pseudocount_threshold)
    print(ad[:, good_genes].layers[raw_layer].shape)
    average_expression_of_good_genes=np.mean(ad[:, good_genes].layers[raw_layer], axis = 0)
    print(average_expression_of_good_genes.shape)

    average_expression_of_good_genes=np.array(average_expression_of_good_genes)
    pseudocount = np.quantile(average_expression_of_good_genes, pseudocount_threshold)
    # pseudocount = np.quantile(np.mean(ad.layers[raw_layer], axis = 0), pseudocount_threshold)
    # pseudocount = np.quantile(np.mean(ad[:, good_genes].layers[raw_layer], axis=1), pseudocount_threshold, axis=0)

    print(pseudocount)
    for cl in np.unique(ad.obs[cl_level]):
        print(cl)

        ## for each gene, number of cells with nonzero counts
        cl_nnz_counts = np.array(np.sum(ad[ad.obs[cl_level] == cl,:].layers[raw_layer] > 0, axis = 0)).squeeze()
        noncl_nnz_counts = np.array(np.sum(ad[ad.obs[cl_level] != cl,:].layers[raw_layer] > 0, axis = 0)).squeeze()

        ## indices for genes where > 5% of cells in the cluster have nonzero counts OR > 5% of cells not in the
        ## cluster have nonzero counts
        gene_is = np.where((cl_nnz_counts >= (cl_pct*(np.sum(ad.obs[cl_level] == cl)))) |
                           (noncl_nnz_counts >= (cl_pct*(np.sum(ad.obs[cl_level] != cl)))))[0]

        ## get Pearson normalized counts for cluster and non cluster cells in these genes
        normalized_cl = ad[ad.obs[cl_level] == cl, gene_is].layers[norm_layer]
        normalized_non_cl = ad[ad.obs[cl_level] != cl,gene_is].layers[norm_layer]

        ## generate size factor, not taking into effect these genes with low expression counts
        cl_size_factor = calculate_size_factor(ad.obs[cl_level] == cl, gene_is)
        noncl_size_factor = calculate_size_factor(ad.obs[cl_level] != cl, gene_is)

        # get pseudobulk counts for genes adjusted by size factor, for cluster cells and non cluster cells
        cl_sums = np.array(np.sum(ad[ad.obs[cl_level] == cl, gene_is].layers[raw_layer],
                                  0)/cl_size_factor).squeeze()
        non_cl_sums = np.array(np.sum(ad[ad.obs[cl_level] != cl, gene_is].layers[raw_layer],
                                      0)/noncl_size_factor).squeeze()
        ## generate LFC in the adjusted pseudobulk counts
        log2FC = np.log2((cl_sums + pseudocount) / (non_cl_sums + pseudocount))
        ## only generate pvals for genes with high LFCs
        log2FC_high = np.where(abs(log2FC) > log2FC_threshold)[0]
        pvals = [mannwhitneyu(normalized_cl[:,i], normalized_non_cl[:,i])[1] for i in range(len(gene_is))]

        # Adjust p-values for multiple testing using Benjamini-Hochberg procedure
        _, adj_p_values, _, _ = mt.multipletests(pvals, method='fdr_bh')
        log2FC_data = {'log2FC': log2FC.tolist(),
                       'cluster_means': cl_sums,
                       'noncluster_means': non_cl_sums,
                       'pval': pvals,
                       'padj':  adj_p_values,
                       'gene': ad.var_names[gene_is]}
        log2FC_data = pd.DataFrame(log2FC_data)
        log2FC_data = log2FC_data.sort_values(by='log2FC', ascending=False)
        print('  results: log2FC > %s: %s genes; log2FC < -%s: %s genes' %
          (log2FC_threshold, sum((log2FC_data['log2FC'] > log2FC_threshold) &
                                 (log2FC_data['padj'] < padj_threshold)),
          log2FC_threshold, sum((log2FC_data['log2FC'] < -log2FC_threshold) &
                                 (log2FC_data['padj'] < padj_threshold))))
        log2FC_data["cluster"] = cl
        # df = df.append(log2FC_data)
        df = pd.concat([df, log2FC_data], axis=0)
    return df


## plotting function for DE genes
def plot_diff_expr(lfc_data, adata, cl_name= "leiden", top_n = 5, figsize=(50, 20), dpi = 100,
                   wspace=0.5, hspace=0.2, nrows=2, color='lightblue', only_up = False):
    print(cl_name)
    fig = plt.figure(figsize=figsize, dpi = dpi)
    plot_clusters = np.unique(lfc_data["cluster"])
    ncols = int(np.ceil(len(plot_clusters) / nrows))
    gs = fig.add_gridspec(nrows=nrows, ncols=ncols, wspace=wspace, hspace=hspace)
    i = 0
    for cluster in np.unique(lfc_data["cluster"]):
            logFC_cluster = lfc_data[lfc_data["cluster"] == cluster]
            genes_up = logFC_cluster[logFC_cluster['log2FC'] > 0].sort_values('log2FC', ascending = False)[:top_n]
            genes_down = logFC_cluster[logFC_cluster['log2FC'] < 0].sort_values('log2FC', ascending = False)[-top_n:]
            genes_both = pd.concat([genes_up, genes_down])
            if only_up:
                genes_both = genes_up
            # print(",".join(genes_up["gene"]))
            ax = plt.subplot(gs[i])
            i += 1
            cluster_size = sum(adata.obs[cl_name] == cluster)
            title = ('cluster %s:\n%s cells' % (cluster, cluster_size))
            if cluster_size >= 1:
                genes_both[::-1].plot(y='log2FC', # figsize=(3, len(genes_both) / 4),
                                      kind='barh',
                                      ax=ax,
                                      width=0.9, color=color, grid=False, legend=False,
                                      title=title)
            plt.xlabel('log2FC')
            plt.ylabel('')
            locs, _ =  ticks = plt.yticks()
            plt.yticks(locs, labels = genes_both[::-1]["gene"])


## find DE ranking for genes of interest
def find_DE_genes(lfcs, cluster, genes_wanted):
    logFC_cluster = lfcs[lfcs["cluster"] == cluster]
    genes_up = logFC_cluster[logFC_cluster['log2FC'] > 0].sort_values('log2FC', ascending = False)
    genes_up.index = range(1,genes_up.shape[0]+1)

    de_gene_list = genes_up['gene'].tolist()
    def intersection(lst1, lst2):
        lst3 = [value for value in lst1 if value in lst2]
        return lst3
    genes_wanted_de = intersection(genes_wanted, de_gene_list)
    print('Genes not found:')
    print(list(set(genes_wanted) - set(genes_wanted_de)))
    print('Genes in de list:')
    genes_wanted_de_df = genes_up[genes_up['gene'].isin(genes_wanted_de)]
    return genes_wanted_de_df

In [ ]:
adata_immune.obs["leiden"] = adata_immune.obs["leiden"].astype(str)

adata_immune.obs["cluster7_vs_other"] = (
    adata_immune.obs["leiden"] == "7"
).astype(int)

adata_immune.obs["cluster7_vs_other"].value_counts()

In [ ]:
adata_immune.obs["leiden"] = adata_immune.obs["leiden"].astype(str)

adata_immune.obs["cluster8_vs_other"] = (
    adata_immune.obs["leiden"] == "8"
).astype(int)

adata_immune.obs["cluster8_vs_other"].value_counts()

In [ ]:
lfcs_Cluster7_or_not = new_lfcs(
    adata_immune,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster7_vs_other"
)

In [ ]:
lfcs_Cluster8_or_not = new_lfcs(
    adata_immune,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="cluster8_vs_other"
)

In [ ]:
plot_diff_expr(lfcs_Cluster7_or_not, adata_immune, figsize=(40, 20), wspace=1, hspace=0.5, nrows=1, only_up=True, top_n = 20,cl_name="cluster7_vs_other")

In [ ]:
plot_diff_expr(lfcs_Cluster8_or_not, adata_immune, figsize=(40, 20), wspace=1, hspace=0.5, nrows=1, only_up=True, top_n = 20,cl_name="cluster8_vs_other")

In [ ]:
stromal_panel = {
    "Endothelial": ["Pecam1","Cdh5","Kdr","Tek"],
    "Smooth_muscle": ["Acta2","Tagln","Myh11"],
    "Fibroblast": ["Pdgfra","Col1a1","Col1a2","Dcn"],
    "Glial": ["Plp1","S100b","Slc1a3"],
    "Immune": ["Ptprc"]
}

sc.pl.dotplot(
    adata_immune,
    stromal_panel,
    groupby="leiden",
    standard_scale="var",
    swap_axes=True
)

In [ ]:
test_markers = {
    "T_cell": ["Cd3d", "Cd3e", "Cd4", "Cd8a", "Trbc1", "Trbc2"],
    "NK_ILC": ["Nkg7", "Ncr1", "Eomes", "Prf1", "Gzmb"],
    "ILC2": ["Il7r", "Rora", "Gata3", "Areg", "Il5", "Il13", "Klrg1", "Il1rl1"],
    "Myeloid": ["Lyz2", "Itgam", "Cd14", "Adgre1", "Csf1r", "Cd68"],
    "Mast": ["Kit", "Tpsb2", "Gata2"]
}

sc.pl.dotplot(
    adata_immune,
    test_markers,
    groupby="leiden",
    standard_scale="var",
    swap_axes=True
)

In [ ]:
#remove cluster 8 as a contaminant
adata_immune = adata_immune[adata_immune.obs["leiden"] != "8"].copy()

In [ ]:
#rerun clustering for immune

In [ ]:
sc.pl.umap(adata_immune, color=['leiden','condition','mouse_id', "Il15", "Il15ra", "Il2rg"
                         ],legend_loc='on data', 
           ncols= 2,  vmin='p5', vmax='p95', wspace = 0.3)

In [ ]:
Immune_Markers = {

    "B_Cells": ["Cd19", "Cd22", "Ms4a1"],

    "T_Cells": ["Cd3d", "Cd3e", "Cd3g", "Cd8a", "Cd4"],
    "NK_cell": ["Nkg7", "Ncr1", "Eomes", "Prf1"],

    "Macrophages": ["Itgam", "Csf1r", "Adgre1", "Itgb1"],
    "Mast_cell": ["Kit", "Tpsb2", "Gata2"],
    "Dendritic": ["Itgax", "Flt3", "Clec9a"],
    "ILC2": ["Il7r", "Rora", "Gata3", "Areg", "Il5", "Il13", "Klrg1", "Il1rl1"],
    "Neutrophils": ["Itgam", "Cd14", "S100a8", "S100a9"]
}

# Dotplot
sc.pl.dotplot(
    adata_immune,
    Immune_Markers,
    groupby="leiden",
    standard_scale="var",
    swap_axes=True
)

# Matrixplot
sc.pl.matrixplot(
    adata_immune,
    Immune_Markers,
    groupby="leiden",
    standard_scale="var",
    swap_axes=True
)



In [ ]:
immunecluster_to_celltype = {
    "0": "NK_cell",
    "1": "Neutrophils",
    "3": "CD4_Tcell",
    "2": "B_Cell",
    "4": "CD8_Tcell",
    "5": "T_cell",
    "6": "Macrophages",
    "7": "ILC2",
    "8": "Macrophages",
    "9": "NK_cell",
    "10": "Mast_cell",
   


}

adata_immune.obs["leiden"] = adata_immune.obs["leiden"].astype(str)
adata_immune.obs["celltype"] = adata_immune.obs["leiden"].map(immunecluster_to_celltype)
adata_immune.obs["celltype"] = adata_immune.obs["celltype"].astype("category")

In [ ]:
sc.pl.umap(adata_immune, color=['leiden','condition','mouse_id',"celltype", "Il15", "Il15ra"
                         ],legend_loc='on data', 
           ncols= 2,  vmin='p5', vmax='p95', wspace = 0.3)

In [ ]:
sc.pl.umap(
    adata_immune,
    color=["celltype","Il2ra","Il2rb","Il2rg","Il15", "Il15ra", "Il1rl1"]
,wspace = 0.4)

# Immune Celltypes by condition 

In [ ]:
lfcs_immune_condition = new_lfcs(
    adata_immune,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="condition"
)

In [ ]:
plot_diff_expr(lfcs_immune_condition, adata_immune, figsize=(40, 20), wspace=1, hspace=0.5, nrows=1, only_up=True, top_n = 20,cl_name="condition")

In [ ]:
lfcs_immune_exercise = lfcs_immune_condition[
    lfcs_immune_condition["cluster"] == "exercise"
].copy()

lfcs_immune_exercise["log2FC"].describe()

In [ ]:
top50 = (
    lfcs_immune_exercise
    .query("padj < 0.05 and log2FC > 0.5")
    .sort_values("log2FC", ascending=False)
    .head(50)
    .reset_index()
    .rename(columns={"index": "gene"})
)

print(top50[["gene", "log2FC", "padj"]])


# Tcell all

In [ ]:
adata_Tcell = adata_immune[adata_immune.obs["leiden"].isin(["3","4","5"])].copy()

print(adata_Tcell.obs["condition"].value_counts())

In [ ]:
lfcs_Tcell = new_lfcs(
    adata_Tcell,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="condition"
)

In [ ]:
plot_diff_expr(lfcs_Tcell, adata_Tcell, figsize=(40, 20), wspace=1, hspace=0.5, nrows=1, only_up=True, top_n = 20,cl_name="condition")

In [ ]:
lfcs_Tcell_exercise = lfcs_Tcell[
    lfcs_Tcell["cluster"] == "exercise"
].copy()

lfcs_Tcell_exercise["log2FC"].describe()

In [ ]:
top50 = (
    lfcs_Tcell_exercise
    .query("padj < 0.05 and log2FC > 0.5")
    .sort_values("log2FC", ascending=False)
    .head(50)
    .reset_index()
    .rename(columns={"index": "gene"})
)

print(top50[["gene", "log2FC", "padj"]])


In [ ]:
top_genes = (
    lfcs_Tcell_exercise
    .sort_values("log2FC", ascending=False)["gene"]
    .head(50)
    .tolist()
)

print(top_genes)

# CD4_Tcell

In [ ]:
adata_CD4Tcell = adata_immune[adata_immune.obs["celltype"] == "CD4_Tcell"].copy()

print(adata_CD4Tcell.obs["condition"].value_counts())

In [ ]:
lfcs_CD4Tcell = new_lfcs(
    adata_CD4Tcell,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="condition"
)

In [ ]:
plot_diff_expr(lfcs_CD4Tcell, adata_CD4Tcell, figsize=(40, 20), wspace=1, hspace=0.5, nrows=1, only_up=True, top_n = 20,cl_name="condition")

   # CD8_Tcell

In [ ]:
adata_CD8Tcell = adata_immune[adata_immune.obs["celltype"] == "CD8_Tcell"].copy()

print(adata_CD8Tcell.obs["condition"].value_counts())

In [ ]:
lfcs_CD8Tcell = new_lfcs(
    adata_CD8Tcell,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="condition"
)

In [ ]:
plot_diff_expr(lfcs_CD8Tcell, adata_CD8Tcell, figsize=(40, 20), wspace=1, hspace=0.5, nrows=1, only_up=True, top_n = 20,cl_name="condition")

# NK cells

In [ ]:
adata_NK_cell = adata_immune[adata_immune.obs["celltype"] == "NK_cell"].copy()

print(adata_NK_cell.obs["condition"].value_counts())

In [ ]:
lfcs_NK_cell = new_lfcs(
    adata_NK_cell,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="condition"
)

In [ ]:
plot_diff_expr(lfcs_NK_cell, adata_NK_cell, figsize=(40, 20), wspace=1, hspace=0.5, nrows=1, only_up=True, top_n = 20,cl_name="condition")

In [ ]:
lfcs_NK_cell_exercise = lfcs_NK_cell[
    lfcs_NK_cell["cluster"] == "exercise"
].copy()

lfcs_NK_cell_exercise["log2FC"].describe()

In [ ]:
top50 = (
    lfcs_NK_cell_exercise
    .query("padj < 0.05 and log2FC > 0.5")
    .sort_values("log2FC", ascending=False)
    .head(50)
    .reset_index()
    .rename(columns={"index": "gene"})
)

print(top50[["gene", "log2FC", "padj"]])


# Bcell

In [ ]:
adata_Bcell = adata_immune[adata_immune.obs["celltype"] == "B_Cell"].copy()

print(adata_Bcell.obs["condition"].value_counts())

In [ ]:
lfcs_Bcell = new_lfcs(
    adata_Bcell,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="condition"
)

In [ ]:
plot_diff_expr(lfcs_Bcell, adata_Bcell, figsize=(40, 20), wspace=1, hspace=0.5, nrows=1, only_up=True, top_n = 20,cl_name="condition")

In [ ]:
lfcs_Bcell_exercise = lfcs_Bcell[
    lfcs_Bcell["cluster"] == "exercise"
].copy()

lfcs_Bcell_exercise["log2FC"].describe()

In [ ]:
lfcs_Bcell_exercise = lfcs_Bcell[
    lfcs_Bcell["cluster"] == "exercise"
].copy()

lfcs_Bcell_exercise["log2FC"].describe()

In [ ]:
top50 = (
    lfcs_Bcell_exercise
    .query("padj < 0.05 and log2FC > 0.5")
    .sort_values("log2FC", ascending=False)
    .head(50)
    .reset_index()
    .rename(columns={"index": "gene"})
)

print(top50[["gene", "log2FC", "padj"]])

# Neutrophils

In [ ]:
adata_Neutrophils = adata_immune[adata_immune.obs["celltype"] == "Neutrophils"].copy()

print(adata_Neutrophils.obs["condition"].value_counts())

In [ ]:
lfcs_Neutrophils = new_lfcs(
    adata_Neutrophils,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="condition"
)

In [ ]:
plot_diff_expr(lfcs_Neutrophils, adata_Neutrophils, figsize=(40, 20), wspace=1, hspace=0.5, nrows=1, only_up=True, top_n = 20,cl_name="condition")

In [ ]:
lfcs_Neutrophils_exercise = lfcs_Neutrophils[
    lfcs_Neutrophils["cluster"] == "exercise"
].copy()

lfcs_Neutrophils_exercise["log2FC"].describe()

In [ ]:
lfcs_Neutrophils_exercise.sort_values("log2FC", ascending=True).head(50)

In [ ]:
top50 = (
    lfcs_Neutrophils_exercise
    .query("padj < 0.05 and log2FC > 0.5")
    .sort_values("log2FC", ascending=False)
    .head(50)
    .reset_index()
    .rename(columns={"index": "gene"})
)

print(top50[["gene", "log2FC", "padj"]])

# Macrophages

In [ ]:
adata_Macrophages = adata_immune[adata_immune.obs["celltype"] == "Macrophages"].copy()

print(adata_Macrophages.obs["condition"].value_counts())

In [ ]:
lfcs_Macrophages = new_lfcs(
    adata_Macrophages,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="condition"
)

In [ ]:
plot_diff_expr(lfcs_Macrophages, adata_Macrophages, figsize=(40, 20), wspace=1, hspace=0.5, nrows=1, only_up=True, top_n = 20,cl_name="condition")

In [ ]:
lfcs_Macrophages_exercise = lfcs_Macrophages[
    lfcs_Macrophages["cluster"] == "exercise"
].copy()

lfcs_Macrophages_exercise["log2FC"].describe()

In [ ]:
lfcs_Macrophages_exercise.sort_values("log2FC", ascending=False).head(50)

In [ ]:
top50 = (
    lfcs_Macrophages_exercise
    .query("padj < 0.05 and log2FC > 0.5")
    .sort_values("log2FC", ascending=False)
    .head(50)
    .reset_index()
    .rename(columns={"index": "gene"})
)

print(top50[["gene", "log2FC", "padj"]])

# ILC2

In [ ]:
adata_ILC2 = adata_immune[adata_immune.obs["celltype"] == "ILC2"].copy()

print(adata_ILC2.obs["condition"].value_counts())

In [ ]:
lfcs_ILC2 = new_lfcs(
    adata_ILC2,
    raw_layer="raw_counts_dense",
    norm_layer="norm_counts_log_dense",
    cl_level="condition"
)

In [ ]:
plot_diff_expr(lfcs_ILC2, adata_ILC2, figsize=(40, 20), wspace=1, hspace=0.5, nrows=1, only_up=True, top_n = 20,cl_name="condition")

In [ ]:
lfcs_ILC2_exercise = lfcs_ILC2[
    lfcs_ILC2["cluster"] == "exercise"
].copy()

lfcs_ILC2_exercise["log2FC"].describe()

In [ ]:
top50 = (
    lfcs_ILC2_exercise
    .query("padj < 0.05 and log2FC > 0.5")
    .sort_values("log2FC", ascending=False)
    .head(50)
    .reset_index()
    .rename(columns={"index": "gene"})
)

print(top50[["gene", "log2FC", "padj"]])

# DE analysis

In [ ]:
from adjustText import adjust_text
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def volcano_plot(df, gene_col="gene", lfc_col="log2FC", padj_col="padj",
                 padj_thresh=0.05, lfc_thresh=0.5, top_n=20,
                 title="Volcano Plot"): 

    df = df.copy()

    # remove common nuisance genes from plotting/labeling
    df = df[~df[gene_col].str.startswith("Gm", na=False)]
    df = df[~df[gene_col].str.contains("Rik", na=False)]
    df = df[~df[gene_col].str.startswith(("Rpl", "Rps", "mt-", "Hbb", "Hba"), na=False)]

    # remove invalid adjusted p-values
    df = df[df[padj_col] > 0].copy()
    df["neg_log10_padj"] = -np.log10(df[padj_col])

    # classify genes
    df["category"] = "Not Significant"
    df.loc[(df[padj_col] < padj_thresh) & (df[lfc_col] > lfc_thresh), "category"] = "Up"
    df.loc[(df[padj_col] < padj_thresh) & (df[lfc_col] < -lfc_thresh), "category"] = "Down"

    # score genes using both effect size and significance
    df["label_score"] = np.abs(df[lfc_col]) * df["neg_log10_padj"]

    colors = {"Up": "#1f77b4", "Down": "#ff7f0e", "Not Significant": "grey"}
    labels = {
        "Up": "Up in Exercise",
        "Down": "Up in Sedentary",
        "Not Significant": "Not Significant"
    }

    plt.figure(figsize=(8, 6))

    # plot each category WITH label for legend
    for cat, color in colors.items():
        subset = df[df["category"] == cat]
        plt.scatter(
            subset[lfc_col],
            subset["neg_log10_padj"],
            s=10,
            color=color,
            alpha=0.6,
            label=labels[cat]   # <-- key addition
        )

    # choose top genes to label
    up = df[df["category"] == "Up"].sort_values("label_score", ascending=False).head(top_n)
    down = df[df["category"] == "Down"].sort_values("label_score", ascending=False).head(top_n)
    label_df = pd.concat([up, down])

    texts = []
    for _, row in label_df.iterrows():
        texts.append(
            plt.text(
                row[lfc_col],
                row["neg_log10_padj"],
                row[gene_col],
                fontsize=8
            )
        )

    adjust_text(
        texts,
        arrowprops=dict(arrowstyle="-", color="black", lw=0.5)
    )

    # threshold lines
    plt.axhline(-np.log10(padj_thresh), linestyle="--")
    plt.axvline(lfc_thresh, linestyle="--")
    plt.axvline(-lfc_thresh, linestyle="--")

    # labels + legend
    plt.xlabel("log2 Fold Change")
    plt.ylabel("-log10 adjusted p-value")
    plt.title(title)
    plt.legend(frameon=False)  # <-- adds legend

    plt.show()

In [ ]:
volcano_plot(lfcs_immune_exercise, title="Volcano Plot – Immune cells (Exercise vs Sedentary)")

In [ ]:
volcano_plot(lfcs_Tcell_exercise, title="Volcano Plot – T cells (Exercise vs Sedentary)")

In [ ]:
volcano_plot(lfcs_NK_cell_exercise, title="Volcano Plot – NK cells (Exercise vs Sedentary)")

In [ ]:
volcano_plot(lfcs_Bcell_exercise, title="Volcano Plot – B cells (Exercise vs Sedentary)")

In [ ]:
volcano_plot(lfcs_ILC2_exercise, title="Volcano Plot – ILC2 cells (Exercise vs Sedentary)")

In [ ]:
volcano_plot(lfcs_Neutrophils_exercise, title="Volcano Plot – Neutrophils (Exercise vs Sedentary)")

In [ ]:
volcano_plot(lfcs_Macrophages_exercise, title="Volcano Plot – Macrophages (Exercise vs Sedentary)")

# Save data

In [ ]:
import pickle

with open("/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC/Micetotal_qc.pkl", "wb") as f:
    pickle.dump(adata, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import pickle

with open("/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC/Miceimmune_qc.pkl", "wb") as f:
    pickle.dump(adata_immune, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import pickle
import numpy as np
import scipy.sparse as sp

payload = {
    "X": adata_immune.X.copy() if not sp.issparse(adata_immune.X) else adata_immune.X.copy(),
    "obs": adata_immune.obs.copy(),
    "var": adata_immune.var.copy(),
    "uns": adata_immune.uns.copy(),
    "obsm": {k: v.copy() for k, v in adata_immune.obsm.items()},
    "varm": {k: v.copy() for k, v in adata_immune.varm.items()},
    "layers": {k: v.copy() for k, v in adata_immune.layers.items()},
}
# optional
if adata_immune.raw is not None:
    payload["raw_X"] = adata_immune.raw.X.copy() if not sp.issparse(adata_immune.raw.X) else adata_immune.raw.X.copy()
    payload["raw_var"] = adata_immune.raw.var.copy()

with open("/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC/Miceimmune_parts.pkl", "wb") as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import pickle

payload = {
    "X": adata_immune.X.copy(),
    "obs": adata_immune.obs.copy(),
    "var_names": adata_immune.var_names.copy(),
}

with open("/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC/Miceimmune_liana_min.pkl", "wb") as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import scipy.sparse as sp
payload = {
    "X": adata_fap.X.copy() if not sp.issparse(adata_fap.X) else adata_fap.X.copy(),
    "obs": adata_fap.obs.copy(),
    "var": adata_fap.var.copy(),
    "uns": adata_fap.uns.copy(),
    "obsm": {k: v.copy() for k, v in adata_fap.obsm.items()},
    "varm": {k: v.copy() for k, v in adata_fap.varm.items()},
    "layers": {k: v.copy() for k, v in adata_fap.layers.items()},
}
# optional
if adata_immune.raw is not None:
    payload["raw_X"] = adata_fap.raw.X.copy() if not sp.issparse(adata_fap.raw.X) else adata_fap.raw.X.copy()
    payload["raw_var"] = adata_fap.raw.var.copy()

with open("/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC/Micefap_parts.pkl", "wb") as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
#load

In [ ]:
import pickle

with open("/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC/Micetotal_qc.pkl", "rb") as f:
    adata = pickle.load(f)

In [ ]:
import pickle

with open("/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC/Miceimmune_liana_min.pkl", "rb") as f:
    adata_immune = pickle.load(f)